## Agents Skills - part 2

Now we are ready to start adding skills. Let's warm up by re-implementing the Titanic analysis from the Agent tutorial.

The usual requirements:

In [ ]:
!pip -q install ollama python-frontmatter

Since this tutorial is not about the core of the agent, the code from part 1 is provided in 'helpers/convenience.py':

In [ ]:
from helpers.convenience import SkilledAgent, show_messages

In [ ]:
OLLAMA_HOST = 'http://10.129.20.4:9090/big'
OLLAMA_MODEL='qwen3:14b' 
sa = SkilledAgent(OLLAMA_HOST, OLLAMA_MODEL, 'skills')
print(list(sa.known_tools))
print(list(sa.skills))

Like before, we'll start by working with the Titanic dataset available in 'work/titanic.csv'

In [ ]:
sa.task("Help me by analysing data in the file 'titanic.csv' from the 'work/' directory. It contains facts about the fate of a subset of the passengers abord the Titanic. How many passenger are listed in the dataset?")

This simple task is likely to fail, in testing I got diferent answers (correct is 887). The reason is that we have not provided any guidance as to how to deal with CSV files or go about the analysis. We'll address that now by adding an 'analysing-data' skill. 

### Adding a skill

Create a folder in 'skills/' and name it 'analysing-data'. Create a file 'SKILL.md' in it with the following content:
~~~
---
name: analysing-data
description: Instructions for loading and analysing data. 
---

## Overview

This skill lets you read and analyse data files.

You can use the python package Pandas to read and analyse data files.

ALWAYS use the './work/' directory for all intermediate files.

### How to read a file into a Pandas dataframe

Use the data loading capabilities of pandas to read a file into a dataframe, but beware of the file format.

Example: Loading CSV data

```python
import pandas as pd

df = pd.read_csv('titanic.csv')
```

## Instructions

1. Gather basic information from the data file

Write a first script to read the data into a dataframe and use 'head()' on the dataframe as a preliminary step to find out column labels and get a grip on the data type stored in each column as a reference for future steps.
Run the script using the 'bash' tool to get the desired output.

Example script: 
```python
import pandas as pd

df = pd.read_csv('titanic.csv')
print(df.head())
```

2. Write analysis script

Taking the result of the previous step into account, write a python script to perform the required analysis and reporting and save it as file using the 'write_file' tool.

3. Run the analysis script

Run the analysis script using the 'bash' tool, e.g. bash("python foo.py")

4. Rendering images in your response

If the analysis script generated an image, ALWAYS use a markdown image link, e.g. ![](work/foo.png), to display it.

~~~

The skill is pretty much self-explanatory, but some points deserve highlighting:

- the "directive" to use Pandas
- the "imperative" to use the 'work/' directory
- the use of examples
- the detailed step-by-step instructions
- the explicit choice of tool in the steps

With that we can start over:

In [ ]:
sa = SkilledAgent(OLLAMA_HOST, OLLAMA_MODEL, 'skills')
list(sa.skills)

N.B. that 'analysing-data' should be listed. If not, make sure the `SKILL.md` file is correctly placed and has the correct content.

Now we can try out the new skill using the same query as above.

_If you get the response "Error: Neither content nor tool calls." at any stage, just rerun the cell._  
_If the agent misbehaves (e.g. avoiding using skills) you must re-instantiate the agent (run cell with `sa = SkilledAgent...`) to reset the message history_

In [ ]:
sa.task("Help me by analysing data in the file 'titanic.csv' from the 'work/' directory. It contains facts about the fate of a subset of the passengers abord the Titanic. How many passenger are listed in the dataset?")

In [ ]:
sa.task("What data is recorded for each passenger, and what is the name of the corresponding column?")

In [ ]:
sa.task("What passenger class was most likely to survive?")

In [ ]:
sa.task("Show me a diagram with likelihood of survival (in percent) versus fare. Bin the fares into suitable ranges, e.g. width of 20 up to 200 and then one bin for all fares above 200.")

To see what is going back and forth, use `show_messages` and examine the files in the 'work/' directory.

In [ ]:
show_messages(sa.messages)

## Dataportal skills

In order to use the data portal we put the WARA-ops data portal access token in the environment. We will use the public dataset 'ERDClogs-parsed' in the following examples, but feel free to experiment with other datasets if you have access.

In [ ]:
token = "" # <-- Paste your WARA-ops data portal token between the quotes

import os
os.environ['DATAPORTAL_CLIENT_TOKEN'] = token

### Adding yet another skill

This step is exactly as before: create a folder in 'skills/' and name it 'accessing-dataportal-data'. Create a file 'SKILL.md' in it with the following content:

~~~
---
name: accessing-dataportal-data
description: Instructions for accessing data from the datasets in the WARA-ops data portal. 
---

## Overview

This skill lets you access data from the datasets in the WARA-ops data portal.

All data access is handled by an instance of DataportalClient available from the python module dataportal.
Credentials will be provided in the environment so you won't need to pass a 'token' parameter.

ALWAYS use the './work/' directory for intermediate files.

For instructions how to analyse data, refer to the skill 'analysing-data'

### How to read a dataset into a Pandas dataframe

Use a DataportalClient instance to access a named dataset

Example: Associating a dataset with the client

```python
from dataportal import DataportalClient

client = DataportalClient()
client.fromDataset('dataset_name')
```

Once the dataset is associated you can list the individual files in the dataset

Example: List the individual files in a dataset

```python
for file_info in client.listFiles():
    print(file_info['FileID'], file_info['StartDate'])
```

Finally, select a datafile by its file ID and read it into a Pandas dataframe

Example:

```python
df = client.getData(fileID=36666)
```

Write and run a suitable script based on the user query.

~~~

In [ ]:
sa = SkilledAgent(OLLAMA_HOST, OLLAMA_MODEL, 'skills')
print(list(sa.skills))
sa.task("Show a list of the contents in the 'ERDClogs-parsed' dataset in the WARA-ops data portal")

In [ ]:
sa.task("Load data for May 29th from 'ERDClogs-parsed' and show the column names.")

In [ ]:
sa.task("List the unique log levels from May 29th")

In [ ]:
sa.task("Normalize 'log_level' values as the uppercased long form, e.g. 'warn', 'warning', 'Warning' should all be 'WARNING'. Save the result in CSV format")

In [ ]:
sa.task('Analyze the CSV file and show me a histogram of the number of warnings by Hostname, with hostname on the y-axis')

In [ ]:
sa.task("Using the data from May 29th, analyze the health of the system and propose actions.")

In [ ]:
sa.task("Using the data from May 29th, generate a status report of the subsystems.")